# 0. Problem
## 550. Game Play Analysis IV — Medium
Calculate the fraction of players who logged in again exactly one day after their first login. Round to 2 decimals.
Official: https://leetcode.com/problems/game-play-analysis-iv/

# 1. Setup

In [ ]:
import pandas as pd
activity_rows=[(1,2,"2016-03-01",5),(1,2,"2016-03-02",6),(2,3,"2017-06-25",1),(3,1,"2016-03-02",0),(3,4,"2018-07-03",5)]
activity_pd=pd.DataFrame(activity_rows,columns=["player_id","device_id","event_date","games_played"]); activity_pd["event_date"]=pd.to_datetime(activity_pd["event_date"]); activity_pd

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
spark=SparkSession.builder.getOrCreate(); activity_spark=spark.createDataFrame(activity_rows,["player_id","device_id","event_date","games_played"]).withColumn("event_date",F.to_date("event_date")); activity_spark.createOrReplaceTempView("Activity")

# 2. SQL Solution

In [ ]:
sql_result=spark.sql("""WITH first_login AS (SELECT player_id,MIN(event_date) AS first_date FROM Activity GROUP BY player_id), retained AS (SELECT DISTINCT f.player_id FROM first_login f JOIN Activity a ON f.player_id=a.player_id AND a.event_date=DATE_ADD(f.first_date,1)) SELECT ROUND((SELECT COUNT(*) FROM retained)*1.0/(SELECT COUNT(*) FROM first_login),2) AS fraction"""); sql_result.show(truncate=False)

# 3. pandas Solution

In [ ]:
first=activity_pd.groupby("player_id",as_index=False).agg(first_date=("event_date","min")); joined=activity_pd.merge(first,on="player_id"); retained=joined.loc[joined["event_date"].eq(joined["first_date"]+pd.Timedelta(days=1)),"player_id"].nunique(); result_pd=pd.DataFrame({"fraction":[round(retained/len(first),2)]}); result_pd

# 4. PySpark Solution

In [ ]:
first=activity_spark.groupBy("player_id").agg(F.min("event_date").alias("first_date")); retained=(first.alias("f").join(activity_spark.alias("a"),(F.col("f.player_id")==F.col("a.player_id"))&(F.col("a.event_date")==F.date_add(F.col("f.first_date"),1))).select(F.col("f.player_id").alias("player_id")).distinct()); result_spark=retained.agg(F.count("*").alias("retained")).crossJoin(first.agg(F.count("*").alias("players"))).select(F.round(F.col("retained")/F.col("players"),2).alias("fraction")); result_spark.show(truncate=False)

# 5. Pattern Mapping
| Concept | SQL | pandas | PySpark |
|---|---|---|---|
| first event | `MIN(date)` | group `min` | group `F.min` |
| next-day retention | join on `first+1` | merge + Timedelta | join + `date_add` |

# 6. Muscle-Memory Round

พิมพ์ใหม่เองโดยไม่ copy คำตอบด้านบน

In [ ]:
# MUSCLE MEMORY — SQL
# Rebuild using temp view(s): Activity

In [ ]:
# MUSCLE MEMORY — PANDAS
# Rebuild using: activity_pd

In [ ]:
# MUSCLE MEMORY — PYSPARK
# Rebuild using: activity_spark